In [8]:
import numpy as np
import matplotlib.pyplot as plt

# Ex 1
with open("Detroit.txt") as f:
    lines = f.readlines()

header_idx = next(
    i for i, line in enumerate(lines)
    if line.strip().startswith("FTP") and "UEMP" in line
)

variables = lines[header_idx].split()

X = np.loadtxt("Detroit.txt", skiprows=header_idx + 1)

print("Shape:", X.shape)
print("Variables:", variables)
print("First five rows:\n", X[:5])

Shape: (13, 14)
Variables: ['FTP', 'UEMP', 'MAN', 'LIC', 'GR', 'CLEAR', 'WM', 'NMAN', 'GOV', 'HE', 'WE', 'HOM', 'ACC', 'ASR']
First five rows:
 [[2.60350e+02 1.10000e+01 4.55500e+02 1.78150e+02 2.15980e+02 9.34000e+01
  5.58724e+05 5.38100e+02 1.33900e+02 2.98000e+00 1.17180e+02 8.60000e+00
  3.91700e+01 3.06180e+02]
 [2.69800e+02 7.00000e+00 4.80200e+02 1.56410e+02 1.80480e+02 8.85000e+01
  5.38584e+05 5.47600e+02 1.37600e+02 3.09000e+00 1.34020e+02 8.90000e+00
  4.02700e+01 3.15160e+02]
 [2.72040e+02 5.20000e+00 5.06100e+02 1.98020e+02 2.09570e+02 9.44000e+01
  5.19171e+05 5.62800e+02 1.43600e+02 3.23000e+00 1.41680e+02 8.52000e+00
  4.53100e+01 2.77530e+02]
 [2.72960e+02 4.30000e+00 5.35800e+02 2.22100e+02 2.31670e+02 9.20000e+01
  5.00457e+05 5.91000e+02 1.50300e+02 3.33000e+00 1.47980e+02 8.89000e+00
  4.95100e+01 2.34070e+02]
 [2.72510e+02 3.50000e+00 5.76000e+02 3.01920e+02 2.97650e+02 9.10000e+01
  4.82418e+05 6.26100e+02 1.64300e+02 3.46000e+00 1.59850e+02 1.30700e+01
  5.5050

In [9]:
X_centered = X - np.mean(X, axis=0)

S = np.cov(X_centered, rowvar=False)

eigenvalues, eigenvectors = np.linalg.eigh(S)

print("S shape:", S.shape)
print("Eigenvalues:", eigenvalues)

S shape: (14, 14)
Eigenvalues: [-1.21718399e-13  3.44269177e-08  1.01359958e-01  3.99093796e-01
  2.58335741e+00  6.30047967e+00  1.59532307e+01  5.92990252e+01
  1.04855146e+02  4.76830970e+02  3.18591840e+03  9.61698806e+03
  5.79992410e+04  4.16919065e+09]


In [10]:
idx = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

explained_variance = eigenvalues / np.sum(eigenvalues)

cumulative_variance = np.cumsum(explained_variance)

for i, variance in enumerate(cumulative_variance):
    print(f"{i+1} PCs: {variance:.6%}")

k80 = np.searchsorted(cumulative_variance, 0.80) + 1
k90 = np.searchsorted(cumulative_variance, 0.90) + 1

print("PCs for 80%:", k80)
print("PCs for 90%:", k90)

1 PCs: 99.998286%
2 PCs: 99.999677%
3 PCs: 99.999908%
4 PCs: 99.999984%
5 PCs: 99.999995%
6 PCs: 99.999998%
7 PCs: 99.999999%
8 PCs: 100.000000%
9 PCs: 100.000000%
10 PCs: 100.000000%
11 PCs: 100.000000%
12 PCs: 100.000000%
13 PCs: 100.000000%
14 PCs: 100.000000%
PCs for 80%: 1
PCs for 90%: 1


In [11]:
k = k90

V_k = eigenvectors[:, :k]
Y = X_centered @ V_k

print("Principal components shape:", Y.shape)
print("Principal component scores:\n", Y)

Principal components shape: (13, 1)
Principal component scores:
 [[-106217.65409818]
 [ -86078.18629021]
 [ -66665.23510682]
 [ -47951.35004486]
 [ -29912.0071357 ]
 [ -12522.63225549]
 [   4241.22243806]
 [  20402.45297766]
 [  35976.18425908]
 [  50990.49897966]
 [  65462.52257514]
 [  79413.63740543]
 [  92860.54629623]]


In [12]:
correlations = np.corrcoef(X.T, Y.T)[:X.shape[1], X.shape[1]:]

for name, corr in zip(variables, correlations[:, 0]):
    print(f"{name}: {corr:.4f}")

FTP: 0.8839
UEMP: -0.0731
MAN: 0.7528
LIC: 0.7837
GR: 0.8458
CLEAR: -0.8906
WM: -1.0000
NMAN: 0.9943
GOV: 0.9886
HE: 0.8667
WE: 0.8603
HOM: 0.9526
ACC: 0.0756
ASR: 0.6531


In [13]:
Z = (X - X.mean(axis=0)) / X.std(axis=0, ddof=1)

S_z = np.cov(Z, rowvar=False)

eigenvalues_z, eigenvectors_z = np.linalg.eigh(S_z)

idx = np.argsort(eigenvalues_z)[::-1]
eigenvalues_z = eigenvalues_z[idx]
eigenvectors_z = eigenvectors_z[:, idx]

explained_z = eigenvalues_z / eigenvalues_z.sum()
cumulative_z = np.cumsum(explained_z)

for i, variance in enumerate(cumulative_z):
    print(f"{i+1} PCs: {variance:.6%}")

k80_z = np.searchsorted(cumulative_z, 0.80) + 1
k90_z = np.searchsorted(cumulative_z, 0.90) + 1

print("PCs for 80%:", k80_z)
print("PCs for 90%:", k90_z)

1 PCs: 68.583969%
2 PCs: 90.638887%
3 PCs: 96.715434%
4 PCs: 98.272780%
5 PCs: 98.931743%
6 PCs: 99.343032%
7 PCs: 99.646444%
8 PCs: 99.802123%
9 PCs: 99.939684%
10 PCs: 99.985293%
11 PCs: 99.997043%
12 PCs: 100.000000%
13 PCs: 100.000000%
14 PCs: 100.000000%
PCs for 80%: 2
PCs for 90%: 2


In [14]:
k = k90_z
V_k_z = eigenvectors_z[:, :k]

Y_z = Z @ V_k_z

print("Principal components shape:", Y_z.shape)
print("Principal component scores:\n", Y_z)

correlations_z = np.corrcoef(X.T, Y_z.T)[:X.shape[1], X.shape[1]:]

print("\nCorrelations:")
for name, corr in zip(variables, correlations_z):
    print(f"{name:6s} PC1: {corr[0]: .4f}  PC2: {corr[1]: .4f}")

Principal components shape: (13, 2)
Principal component scores:
 [[ 3.81266544  3.16041727]
 [ 3.29992982  2.08722453]
 [ 3.18418773  0.64927661]
 [ 2.8394824  -0.4774624 ]
 [ 2.08073994 -1.72984271]
 [ 1.49602652 -2.12465088]
 [ 0.47186596 -1.43903892]
 [-0.4585438  -2.32942717]
 [-1.64743442 -1.54329216]
 [-2.19114016  0.10713261]
 [-3.32135034  1.08678315]
 [-4.40366091  1.60923336]
 [-5.1627682   0.9436467 ]]

Correlations:
FTP    PC1: -0.9533  PC2:  0.2371
UEMP   PC1: -0.1117  PC2:  0.9152
MAN    PC1: -0.6240  PC2: -0.7512
LIC    PC1: -0.7313  PC2: -0.4114
GR     PC1: -0.8287  PC2: -0.2342
CLEAR  PC1:  0.9566  PC2: -0.2338
WM     PC1:  0.9760  PC2:  0.2016
NMAN   PC1: -0.9784  PC2: -0.1799
GOV    PC1: -0.9734  PC2: -0.1576
HE     PC1: -0.9196  PC2:  0.2075
WE     PC1: -0.9043  PC2:  0.1465
HOM    PC1: -0.9928  PC2:  0.0826
ACC    PC1:  0.1285  PC2: -0.9570
ASR    PC1: -0.7947  PC2:  0.5156
